Q8 - GMM fitted on latents of VQVAE and outputs plotted

In [8]:
from sklearn.mixture import GaussianMixture
import numpy as np

# Loading the latent vectors obtained from posterior inference
latent_vectors = np.load("latent_representations.npy")

# Fitting a GMM to the latent vectors
n_components = 10  
gmm = GaussianMixture(n_components=n_components, covariance_type='diag', random_state=42)
gmm.fit(latent_vectors)

print("GMM fitted to latent vectors.")

GMM fitted to latent vectors.


In [ ]:
import torch
import torch.nn as nn
# Sample 100 new latent vectors from the GMM
n_samples =100
sampled_latents = gmm.sample(n_samples)[0]  # [0] to get the samples (ignore the component labels)




In [11]:
class ResidualLayer(nn.Module):
    def __init__(self, res_h_dim):
        super(ResidualLayer, self).__init__()
        self.block = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(res_h_dim, res_h_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(res_h_dim, res_h_dim, kernel_size=1)
        )

    def forward(self, x):
        return x + self.block(x)  # Residual connection


class ResidualStack(nn.Module):
    def __init__(self, res_h_dim, n_res_layers):
        super(ResidualStack, self).__init__()
        self.residual_layers = nn.ModuleList([ResidualLayer(res_h_dim) for _ in range(n_res_layers)])

    def forward(self, x):
        for layer in self.residual_layers:
            x = layer(x)
        return x

class Encoder(nn.Module):
    def __init__(self, in_channels, h_dim, res_h_dim, n_res_layers):
        super(Encoder, self).__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(in_channels, h_dim // 2, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(h_dim // 2, h_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(h_dim, res_h_dim, kernel_size=4, stride=2, padding=1)
        )
        self.residual_stack = ResidualStack(res_h_dim, n_res_layers)

    def forward(self, x):
        x = self.initial(x)
        x = self.residual_stack(x)
        return x


class Decoder(nn.Module):
    def __init__(self, out_channels, h_dim, res_h_dim, n_res_layers):
        super(Decoder, self).__init__()
        self.residual_stack = ResidualStack(res_h_dim, n_res_layers)
        self.initial = nn.Sequential(
            nn.ConvTranspose2d(res_h_dim, h_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(h_dim, h_dim // 2, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(h_dim // 2, out_channels, kernel_size=4, stride=2, padding=1)
        )

    def forward(self, x):
        x = self.residual_stack(x)
        x = self.initial(x)
        return x
    
class VectorQuantizerEMA(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25, decay=0.99, epsilon=1e-5):
        super(VectorQuantizerEMA, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.decay = decay
        self.epsilon = epsilon

        # Initialize embeddings
        self.embeddings = nn.Parameter(torch.randn(num_embeddings, embedding_dim))
        self.register_buffer("ema_cluster_size", torch.zeros(num_embeddings))
        self.register_buffer("ema_embedding_avg", torch.randn(num_embeddings, embedding_dim))

    def forward(self, x):
        # Flatten input to (batch_size * num_latents, embedding_dim)
        flat_x = x.view(-1, self.embedding_dim)

        # Compute distances between encoder outputs and embeddings
        distances = (
            torch.sum(flat_x ** 2, dim=1, keepdim=True)
            + torch.sum(self.embeddings ** 2, dim=1)
            - 2 * torch.matmul(flat_x, self.embeddings.t())
        )

        # Find nearest embedding index for each latent vector
        encoding_indices = torch.argmin(distances, dim=1)
        quantized = self.embeddings[encoding_indices].view(x.shape)

        # Compute commitment loss
        e_latent_loss = torch.mean((quantized.detach() - x) ** 2)
        loss = self.commitment_cost * e_latent_loss

        # EMA updates for the embeddings
        if self.training:
            # One-hot encode indices
            encodings = torch.zeros(encoding_indices.size(0), self.num_embeddings, device=x.device)
            encodings.scatter_(1, encoding_indices.unsqueeze(1), 1)

            # Update cluster size using EMA
            ema_cluster_size = self.ema_cluster_size * self.decay + (1 - self.decay) * encodings.sum(0)
            # Laplace smoothing to avoid division by zero
            n = torch.sum(ema_cluster_size) + self.epsilon
            self.ema_cluster_size = ema_cluster_size / n * n  # Normalized by overall cluster size

            # Update embedding averages using EMA
            embedding_sum = torch.matmul(encodings.t(), flat_x)
            self.ema_embedding_avg = self.ema_embedding_avg * self.decay + (1 - self.decay) * embedding_sum

            # Normalize embedding averages to get updated codebook
            self.embeddings.data = self.ema_embedding_avg / (self.ema_cluster_size.unsqueeze(1) + self.epsilon)

        # Detach gradients from quantized output for backprop
        quantized = x + (quantized - x).detach()

        return quantized, loss

class VQVAE(nn.Module):
    def __init__(self, in_channels, out_channels, h_dim, res_h_dim, n_res_layers, num_embeddings):
        super(VQVAE, self).__init__()
        self.encoder = Encoder(in_channels, h_dim, res_h_dim, n_res_layers)
        self.decoder = Decoder(out_channels, h_dim, res_h_dim, n_res_layers)
        self.vector_quantizer = VectorQuantizerEMA(num_embeddings, res_h_dim)  # Using EMA-based quantizer

    def forward(self, x):
        encoded = self.encoder(x)
        quantized, vq_loss = self.vector_quantizer(encoded)
        decoded = self.decoder(quantized)
        return decoded, vq_loss

In [ ]:
vqvae = torch.load('VQ_VAE-EMA_res.pth',weights_only=False)
vqvae.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vqvae.to(device)

# Generating images from sampled latents using the decoder
with torch.no_grad():
    generated_images = vqvae.decoder(sampled_latents_tensor)


In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

# Move generated images to CPU and create a 10x10 grid
generated_images = generated_images.cpu()
grid = make_grid(generated_images, nrow=10, normalize=True)

# Plot the grid of generated images
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0))  # Change dimensions for plotting (C, H, W) -> (H, W, C)
plt.axis("off")
plt.show()
